# Retail Sales & Profitability Analysis (EDA)

**Project type:** Exploratory Data Analysis (EDA)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Currency:** KES (Kenyan shillings)

## Objective
Explore retail order data to understand sales and profitability trends, compare product categories, examine customer performance, and investigate how discounts relate to profit.

## Questions explored
- What data-quality issues need to be addressed?
- Which categories and sub-categories generate the most sales and profit?
- How do sales change over time?
- Which customers and products contribute most to profit or losses?
- How are discount levels associated with profitability?

> **Note:** This notebook expects `retail-orders.csv` to be available in the same folder. Add the dataset to the repository only if you have permission to share it. Otherwise, explain how to obtain it in the repository README.

## 1. Import libraries and load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("retail-orders.csv")
df.head()


## 2. Initial data inspection

In [ ]:
df.shape


In [ ]:
df.columns


In [ ]:
df.info()


## 3. Data quality: missing values and duplicates

In [ ]:
df.isnull().sum()


In [ ]:
df[df['town'].isnull()]


In [ ]:
df.duplicated().sum()


In [ ]:
df[df.duplicated()]


In [ ]:
df[df.duplicated(keep=False)].sort_values(by='row_id')


In [ ]:
df['row_id'].duplicated().sum()


In [ ]:
df[df['row_id'].duplicated(keep=False)].sort_values('row_id')


In [ ]:
df = df.drop_duplicates()


In [ ]:
df.shape


In [ ]:
df['town'] = df['town'].fillna('Unknown')


In [ ]:
df['town'].isnull().sum()


## 4. Descriptive statistics and extreme losses

In [ ]:
df.describe()


In [ ]:
df.sort_values(by="profit_kes").head(10)


In [ ]:
df[df['discount'] == 0.7][['product_name', 'sales_kes', 'discount', 'profit_kes']].head(10)


## 5. Discount and profitability analysis

In [ ]:
df[df['discount'] == 0.7]['profit_kes'].agg(['count', 'min', 'max', 'mean'])


In [ ]:
df.groupby('discount')['profit_kes'].agg(['count', 'mean', 'min', 'max']).round(2)


In [ ]:
df.groupby('discount')['profit_kes'].agg(['count', 'min', 'max', 'mean']).round(2)


## 6. Category-level sales and profit

In [ ]:
df.groupby('category')['profit_kes'].agg(['sum', 'mean']).sort_values(by='sum', ascending=False).round(2)


In [ ]:
df.groupby('category')['sales_kes'].agg(['sum','mean']).sort_values(by='sum',ascending=False).round(2)


In [ ]:
category_sales = df.groupby('category')['sales_kes'].sum()

category_sales.plot(kind='bar')

plt.title('Total Sales by Category')
plt.xlabel('Category')
plt.ylabel('Total Sales (KES)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
category_profit = df.groupby('category')['profit_kes'].sum()

category_profit.plot(kind='bar')

plt.title('Total Profit by Category')
plt.xlabel('Category')
plt.ylabel('Total Profit (KES)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Sales trends over time

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])


In [ ]:
df['year_month'] = df['order_date'].dt.to_period('M')


In [ ]:
monthly_sales = df.groupby('year_month')['sales_kes'].sum()

monthly_sales


In [ ]:
monthly_sales.sort_values(ascending=False).head(5)


In [ ]:
monthly_sales.sort_index().plot(kind='line', marker='o', figsize=(12, 5))

plt.title('Monthly Sales Trend')
plt.xlabel('Year-Month')
plt.ylabel('Total Sales (KES)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
df['year'] = df['order_date'].dt.year


In [ ]:
yearly_sales = df.groupby('year')['sales_kes'].sum()

print(yearly_sales)


In [ ]:
yearly_sales.plot(kind='bar')

plt.title('Year-wise Sales Comparison')
plt.xlabel('Year')
plt.ylabel('Total Sales (KES)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Customer performance

In [ ]:
top_customers = df.groupby('customer_name')['sales_kes'].sum()

top_customers = top_customers.sort_values(ascending=False).head(10)

print(top_customers)


In [ ]:
top_customers.plot(kind='bar', figsize=(10, 5))

plt.title('Top 10 Customers by Sales')
plt.xlabel('Customer Name')
plt.ylabel('Total Sales (KES)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
top_profit_customers = (
    df.groupby('customer_name')['profit_kes']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_profit_customers)


In [ ]:
customer_analysis = df.groupby('customer_name').agg(
    total_sales=('sales_kes', 'sum'),
    total_profit=('profit_kes', 'sum')
)

customer_analysis = customer_analysis.sort_values(
    by='total_sales', ascending=False
)

customer_analysis.head(10)


In [ ]:
customer_analysis['profit_margin_%'] = (
    customer_analysis['total_profit']
    / customer_analysis['total_sales']
) * 100

customer_analysis.head(10).round(2)


In [ ]:
customer_analysis.sort_values(
    by='profit_margin_%',
    ascending=False
).head(10).round(2)


In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    customer_analysis['total_sales'],
    customer_analysis['total_profit']
)

plt.title('Customer Sales vs Profit')
plt.xlabel('Total Sales (KES)')
plt.ylabel('Total Profit (KES)')
plt.tight_layout()
plt.show()


In [ ]:
correlation = customer_analysis[
    ['total_sales', 'total_profit']
].corr()

print(correlation)


In [ ]:
loss_customers = customer_analysis[
    customer_analysis['total_profit'] < 0
].sort_values(by='total_profit')

print(loss_customers.head(10))


In [ ]:
njeri_orders = df[
    df['customer_name'] == 'Njeri Mutua'
]

njeri_orders[
    [
        'order_id',
        'order_date',
        'product_name',
        'sales_kes',
        'quantity',
        'discount',
        'profit_kes'
    ]
].sort_values(by='profit_kes').head(10)


## 9. Overall discount analysis

In [ ]:
df.groupby('discount').agg(
    total_sales=('sales_kes', 'sum'),
    total_profit=('profit_kes', 'sum'),
    total_orders=('order_id', 'nunique')
).round(2)


In [ ]:
discount_analysis = df.groupby('discount').agg(
    total_sales=('sales_kes', 'sum'),
    total_profit=('profit_kes', 'sum')
)

discount_analysis['profit_margin_%'] = (
    discount_analysis['total_profit']
    / discount_analysis['total_sales']
) * 100

discount_analysis.round(2)


In [ ]:
discount_analysis['profit_margin_%'].plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title('Discount vs Profit Margin')
plt.xlabel('Discount')
plt.ylabel('Profit Margin (%)')
plt.axhline(y=0, linestyle='--')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. Category profitability across discount levels

In [ ]:
category_discount = df.pivot_table(
    index='category',
    columns='discount',
    values='profit_kes',
    aggfunc='mean'
)

category_discount.round(2)


In [ ]:
plt.figure(figsize=(12, 5))

sns.heatmap(
    category_discount,
    annot=True,
    fmt='.0f',
    cmap='RdYlGn',
    center=0
)

plt.title('Category vs Discount: Average Profit')
plt.xlabel('Discount')
plt.ylabel('Category')
plt.tight_layout()
plt.show()


## 11. Profit margin by category

In [ ]:
category_profit = df.groupby('category').agg(
    total_sales=('sales_kes', 'sum'),
    total_profit=('profit_kes', 'sum')
)

category_profit['profit_margin_%'] = (
    category_profit['total_profit']
    / category_profit['total_sales']
) * 100

category_profit.sort_values(
    by='total_profit',
    ascending=True
).round(2)


In [ ]:
category_profit['profit_margin_%'].sort_values().plot(
    kind='bar',
    figsize=(8, 5)
)

plt.title('Profit Margin by Category')
plt.xlabel('Category')
plt.ylabel('Profit Margin (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. Sub-category profitability

In [ ]:
subcategory_analysis = df.groupby('sub_category').agg(
    total_sales=('sales_kes', 'sum'),
    total_profit=('profit_kes', 'sum')
)

subcategory_analysis['profit_margin_%'] = (
    subcategory_analysis['total_profit']
    / subcategory_analysis['total_sales']
) * 100

subcategory_analysis = subcategory_analysis.sort_values(
    by='total_profit',
    ascending=True
)

subcategory_analysis.round(2)


In [ ]:
problem_categories = df[
    df['sub_category'].isin(['Tables', 'Bookcases'])
]

discount_subcategory = problem_categories.pivot_table(
    index='sub_category',
    columns='discount',
    values='profit_kes',
    aggfunc='mean'
)

discount_subcategory.round(2)


## 13. Investigating loss-making table products

In [ ]:
tables_loss = (
    df[df['sub_category'] == 'Tables']
    .groupby('product_name')
    .agg(
        total_sales=('sales_kes', 'sum'),
        total_profit=('profit_kes', 'sum'),
        total_quantity=('quantity', 'sum')
    )
    .sort_values(by='total_profit')
)

tables_loss.head(10)


In [ ]:
top_loss_products = [
    'Tausi Coffee Table',
    'Kifaru Folding Table',
    'Swara Computer Desk Model 2',
    'Duma Computer Desk',
    'Pundamilia Writing Desk'
]

df[df['product_name'].isin(top_loss_products)].groupby(
    'product_name'
).agg(
    total_sales=('sales_kes', 'sum'),
    total_profit=('profit_kes', 'sum'),
    avg_discount=('discount', 'mean'),
    total_quantity=('quantity', 'sum')
).round(2).sort_values(by='total_profit')


In [ ]:
df[df['product_name'].isin(top_loss_products)].groupby(
    'product_name'
).agg(
    total_records=('profit_kes', 'count'),
    profitable_records=('profit_kes', lambda x: (x > 0).sum()),
    loss_records=('profit_kes', lambda x: (x < 0).sum()),
    min_profit=('profit_kes', 'min'),
    max_profit=('profit_kes', 'max')
)


## 14. Deep dive: Tausi Coffee Table

In [ ]:
df[
    (df['product_name'] == 'Tausi Coffee Table') &
    (df['profit_kes'] < 0)
][
    ['order_id', 'order_date', 'sales_kes',
     'quantity', 'discount', 'profit_kes']
].sort_values(by='profit_kes').head(10)


In [ ]:
tausi_analysis = (
    df[df['product_name'] == 'Tausi Coffee Table']
    .groupby('quantity')
    .agg(
        total_records=('profit_kes', 'count'),
        total_sales=('sales_kes', 'sum'),
        total_profit=('profit_kes', 'sum'),
        avg_discount=('discount', 'mean')
    )
)

tausi_analysis['profit_margin_%'] = (
    tausi_analysis['total_profit'] /
    tausi_analysis['total_sales']
) * 100

tausi_analysis.round(2)


In [ ]:
tausi_discount_analysis = (
    df[df['product_name'] == 'Tausi Coffee Table']
    .groupby('discount')
    .agg(
        records=('profit_kes', 'count'),
        total_sales=('sales_kes', 'sum'),
        total_profit=('profit_kes', 'sum'),
        avg_quantity=('quantity', 'mean')
    )
)

tausi_discount_analysis['profit_margin_%'] = (
    tausi_discount_analysis['total_profit'] /
    tausi_discount_analysis['total_sales']
) * 100

tausi_discount_analysis.round(2)


In [ ]:
tables_discount_analysis = (
    df[df['sub_category'] == 'Tables']
    .groupby('discount')
    .agg(
        records=('profit_kes', 'count'),
        total_sales=('sales_kes', 'sum'),
        total_profit=('profit_kes', 'sum')
    )
)

tables_discount_analysis['profit_margin_%'] = (
    tables_discount_analysis['total_profit'] /
    tables_discount_analysis['total_sales']
) * 100

tables_discount_analysis.round(2)


## 15. Key findings

Based on the analyses in this notebook:

1. **Data quality:** Exact duplicate rows were removed and missing `town` values were filled with `Unknown`.
2. **Category performance:** Technology generated the highest total sales and profit in the analysed data.
3. **Furniture profitability:** Furniture had substantial sales but a much lower overall profit margin than the other main categories.
4. **Discounts and profit:** Higher discount groups were associated with lower aggregate profit margins; the Tables sub-category showed especially large losses at higher discount levels.
5. **Product investigation:** Tausi Coffee Table and several other table products had negative aggregate profit. Some products also had profitable records, so performance may vary by transaction.

## Limitations and next steps

- These are descriptive findings. They show associations, not proof that discounts alone caused losses.
- Some product-level groups contain few records, so their results should be interpreted cautiously.
- A useful next step would be to validate pricing, costs, discount policies, and order-level details with the business team.
- For a portfolio-ready project, add selected visualizations and a short README describing the dataset source and how to run the notebook.